# 03 — Prediksi test dan pemeriksaan submission

Notebook ini menjelaskan inferensi dari checkpoint DiT-large dan memeriksa file yang benar-benar dikirim ke kompetisi. Tidak ada perintah upload otomatis. Label test tidak tersedia, sehingga kualitas test dinilai oleh Kaggle.

In [1]:
from pathlib import Path
import sys

# Notebook dapat dijalankan dari root repository atau folder notebooks/.
ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError("Jalankan notebook dari root repository atau folder notebooks/.")
DATA_DIR = ROOT / "data"
sys.path.insert(0, str(ROOT / "src"))

import hashlib
import os
import subprocess
import pandas as pd
from IPython.display import display

## 1. File submission historis

File `submissions/dit_large_v2_44d3276e318d.csv` adalah artefak yang dikirim sebagai ref `57871182`. Format wajib: `id,label`; label berupa integer 0–15, satu baris untuk setiap ID pada `test.csv`, dengan urutan yang sama.

In [2]:
test = pd.read_csv(DATA_DIR / "test.csv")
SUBMISSION_PATH = ROOT / "submissions" / "dit_large_v2_44d3276e318d.csv"
submission = pd.read_csv(SUBMISSION_PATH)
display(submission.head())
print("Baris test/submission:", len(test), len(submission))
print("SHA-256:", hashlib.sha256(SUBMISSION_PATH.read_bytes()).hexdigest())

,id,label
0,test_00001,0
1,test_00002,0
2,test_00003,1
3,test_00004,4
4,test_00005,13


Baris test/submission: 7994 7994
SHA-256: 44d3276e318d3605575e39d9ec42fc2af7256dd30a9b9e0905fb16b34e90ac7d


In [3]:
assert submission.columns.tolist() == ["id", "label"]
assert len(submission) == len(test) == 7994
assert submission.id.is_unique and submission.id.equals(test.id)
assert not submission.isna().any().any()
assert pd.api.types.is_integer_dtype(submission.label)
assert submission.label.between(0, 15).all()
print("Format, urutan ID, jumlah baris, dan rentang label sesuai.")
display(submission.label.value_counts().sort_index().rename_axis("label").to_frame("prediksi"))

Format, urutan ID, jumlah baris, dan rentang label sesuai.


,prediksi
label,
0,490
1,486
2,507
3,516
4,522
5,524
6,483
7,512
8,469


## 2. Inferensi ulang (opsional)

Jika notebook modeling dijalankan ulang, arahkan `CHECKPOINT_PATH` ke checkpoint baru. Untuk menggunakan bobot run historis, unduh `checkpoint/best.pt` dari output kernel Kaggle `muhammadkafanalkafi/document-classification-dit-large-v2` lalu ubah path di bawah. Checkpoint tidak disimpan di Git karena berukuran besar. Inferensi ulang membutuhkan dataset test dan dependensi yang sama; dengan GPU proses akan lebih cepat.

In [4]:
RUN_INFERENCE = False
CHECKPOINT_PATH = ROOT / "checkpoint" / "notebook_dit_large" / "best.pt"
NEW_SUBMISSION = ROOT / "checkpoint" / "notebook_dit_large" / "submission.csv"
predict_command = [sys.executable, str(ROOT / "scripts" / "predict.py"),
                   "--data-dir", str(DATA_DIR), "--checkpoint", str(CHECKPOINT_PATH),
                   "--output", str(NEW_SUBMISSION), "--batch-size", "32", "--workers", "2"]
print("Perintah:", " ".join(predict_command))
if RUN_INFERENCE:
    if not CHECKPOINT_PATH.is_file():
        raise FileNotFoundError(f"Checkpoint tidak ditemukan: {CHECKPOINT_PATH}")
    env = os.environ.copy()
    env["PYTHONPATH"] = str(ROOT / "src") + os.pathsep + env.get("PYTHONPATH", "")
    subprocess.run(predict_command, cwd=ROOT, env=env, check=True)
    new_submission = pd.read_csv(NEW_SUBMISSION)
    assert new_submission.columns.tolist() == ["id", "label"]
    assert len(new_submission) == len(test) and new_submission.id.equals(test.id)
    assert new_submission.id.is_unique and not new_submission.isna().any().any()
    assert pd.api.types.is_integer_dtype(new_submission.label) and new_submission.label.between(0, 15).all()
    print("Prediksi baru valid:", NEW_SUBMISSION)
else:
    print("Inferensi ulang tidak dijalankan; file historis telah diperiksa di atas.")

Perintah: /home/voyager/.t3/projects/sml-cnn-model/.venv/bin/python /home/voyager/.t3/projects/sml-cnn-model/scripts/predict.py --data-dir /home/voyager/.t3/projects/sml-cnn-model/data --checkpoint /home/voyager/.t3/projects/sml-cnn-model/checkpoint/notebook_dit_large/best.pt --output /home/voyager/.t3/projects/sml-cnn-model/checkpoint/notebook_dit_large/submission.csv --batch-size 32 --workers 2
Inferensi ulang tidak dijalankan; file historis telah diperiksa di atas.


## 3. Hasil dan cara submit

Submission historis ref `57871182` memperoleh public score **0,89753**. Pada pemeriksaan 2 November 2026 posisi akun berada di peringkat 3; peringkat berubah mengikuti submission peserta lain. Private score tidak diverifikasi di sini.

Untuk kompetisi yang masih menerima submission, periksa aturan dan batas harian, lalu kirim secara manual dengan Kaggle CLI sesuai [workflow.md](../workflow.md). File notebook ini tidak mengirim hasil ke Kaggle atau formulir pengumpulan.